# ♻️ EcoExpert — Sistema Especialista de Resíduos Domésticos

**Disciplina:** Inteligência Artificial e Linguagens Formais e Autômatos (LFA) — UDF Centro Universitário  
**Docente:** Profa. Kadidja Valéria  

Este notebook reúne a implementação completa do protótipo **EcoExpert** em Python, incluindo:
1. **Base de conhecimento** (21 resíduos em 4 grupos) e **normalização textual**;
2. **Motor de inferência determinístico** com regras de produção (`R1` a `R5`);
3. **Máquina de estados finita** ($Q = \{\text{MENU}, \text{CLASSIFICAR}, \text{HISTORICO}, \text{FIM}\}$);
4. **Testes automatizados (`unittest`)** e **validações de fluxo**.


## 1. Base de Conhecimento, Regras e Normalização

A entrada textual passa pela função `normalizar()`, que remove acentos, converte letras para minúsculas e ajusta espaços extras. A base de conhecimento contém **21 itens** distribuídos em quatro grupos.


In [ ]:
import unicodedata
from typing import Dict, List, Tuple

BASE: Dict[str, List[str]] = {
    "especial": [
        "pilha",
        "bateria",
        "lampada fluorescente",
        "oleo de cozinha",
        "remedio vencido",
        "eletronico",
    ],
    "reciclavel": [
        "papel",
        "papelao",
        "garrafa pet",
        "lata de aluminio",
        "vidro",
        "plastico",
    ],
    "organico": [
        "borra de cafe",
        "casca de fruta",
        "resto de comida",
        "folha seca",
        "casca de ovo",
    ],
    "rejeito": [
        "fralda",
        "papel higienico",
        "absorvente",
        "bituca de cigarro",
    ],
}

REGRAS: Tuple[Tuple[str, str, str, str], ...] = (
    (
        "R1",
        "especial",
        "Descarte especial",
        "Encaminhe a um ponto de coleta específico ou logística reversa. Não descarte no lixo comum.",
    ),
    (
        "R2",
        "reciclavel",
        "Reciclável",
        "Lave levemente se necessário, seque e encaminhe para a coleta seletiva.",
    ),
    (
        "R3",
        "organico",
        "Orgânico",
        "Destine para compostagem doméstica/comunitária ou coleta de resíduos orgânicos.",
    ),
    (
        "R4",
        "rejeito",
        "Rejeito",
        "Descarte no lixo comum (rejeito), pois não é reciclável nem compostável nas condições habituais.",
    ),
)

ORIENTACAO_DESCONHECIDO = (
    "Resíduo não encontrado na base de conhecimento. "
    "Consulte o serviço de limpeza urbana ou a coleta seletiva do seu município."
)


def normalizar(texto: str) -> str:
    """Normaliza o texto removendo acentos, convertendo para minúsculas e ajustando espaços."""
    sem_acento = "".join(
        c for c in unicodedata.normalize("NFKD", texto) if not unicodedata.combining(c)
    )
    return " ".join(sem_acento.lower().split())


print(f"Base carregada com {sum(len(v) for v in BASE.values())} resíduos em {len(BASE)} grupos.")


## 2. Motor de Inferência e Máquina de Estados

O motor `inferir()` avalia as regras em ordem de prioridade (`R1` → `R4`, com `R5` como fallback para entradas desconhecidas ou vazias). A função `proximo_estado()` controla as transições do menu.


In [ ]:
def inferir(entrada: str) -> Tuple[str, str, str]:
    """Retorna uma tupla (categoria, orientacao, regra_aplicada)."""
    termo = normalizar(entrada)
    if not termo:
        return ("Não identificado", ORIENTACAO_DESCONHECIDO, "R5")

    for codigo_regra, grupo, categoria, orientacao in REGRAS:
        if termo in BASE[grupo]:
            return (categoria, orientacao, codigo_regra)

    return ("Não identificado", ORIENTACAO_DESCONHECIDO, "R5")


def proximo_estado(estado_atual: str, evento: str) -> str:
    """Função de transição da máquina de estados finita do menu."""
    if estado_atual == "MENU":
        transicoes = {
            "1": "CLASSIFICAR",
            "2": "HISTORICO",
            "3": "FIM",
        }
        return transicoes.get(evento.strip(), "MENU")
    if estado_atual in ("CLASSIFICAR", "HISTORICO"):
        return "MENU"
    return "FIM"


## 3. Testes Automatizados (`unittest`)

Execução da suíte de testes unitários diretamente no Colab para validar classificação, regras, normalização, tamanho da base e máquina de estados.


In [ ]:
import unittest


class TestInferencia(unittest.TestCase):
    def test_categorias(self):
        casos = [
            ("PAPEL", "Reciclável"),
            ("  borra de café  ", "Orgânico"),
            ("fralda", "Rejeito"),
            ("pilha", "Descarte especial"),
            ("objeto desconhecido", "Não identificado"),
            ("", "Não identificado"),
            ("garrafa   PET", "Reciclável"),
        ]
        for entrada, esperado in casos:
            with self.subTest(entrada=entrada):
                self.assertEqual(inferir(entrada)[0], esperado)

    def test_regras_e_orientacoes(self):
        self.assertEqual(inferir("pilha")[2], "R1")
        self.assertEqual(inferir("papel")[2], "R2")
        self.assertEqual(inferir("borra de café")[2], "R3")
        self.assertEqual(inferir("fralda")[2], "R4")
        self.assertEqual(inferir("item inexistente")[2], "R5")
        self.assertTrue(len(inferir("papel")[1]) > 0)

    def test_normalizacao(self):
        self.assertEqual(normalizar("  Lâmpada   Fluorescente  "), "lampada fluorescente")
        self.assertEqual(normalizar("Óleo de Cozinha"), "oleo de cozinha")
        self.assertEqual(normalizar("   "), "")

    def test_base_conhecimento_21_itens(self):
        total = sum(len(itens) for itens in BASE.values())
        self.assertEqual(total, 21)


class TestMaquinaDeEstados(unittest.TestCase):
    def test_transicoes_menu(self):
        self.assertEqual(proximo_estado("MENU", "1"), "CLASSIFICAR")
        self.assertEqual(proximo_estado("MENU", "2"), "HISTORICO")
        self.assertEqual(proximo_estado("MENU", "3"), "FIM")
        self.assertEqual(proximo_estado("MENU", "9"), "MENU")

    def test_retorno_ao_menu(self):
        self.assertEqual(proximo_estado("CLASSIFICAR", "concluida"), "MENU")
        self.assertEqual(proximo_estado("HISTORICO", "concluida"), "MENU")


unittest.main(argv=["ignored"], exit=False, verbosity=2)


## 4. Validações Detalhadas e Simulação de Fluxo

Abaixo validamos visualmente todos os 7 casos do relatório técnico, mais exemplos adicionais e a simulação completa do fluxo de navegação (`MENU` → `1` → `papel` → `2` → `3` → `FIM`).


In [ ]:
casos_validacao = [
    "PAPEL",
    "  borra de café  ",
    "fralda",
    "pilha",
    "objeto desconhecido",
    "",
    "garrafa   PET",
    "Lâmpada Fluorescente",
    "óleo de cozinha",
    "casca de fruta",
    "bituca de cigarro",
]

print(f"{'ENTRADA':<25} | {'NORMALIZADO':<22} | {'REGRA':<5} | {'CATEGORIA'}")
print("-" * 80)
for item in casos_validacao:
    norm = normalizar(item) or "(vazio)"
    categoria, _, regra = inferir(item)
    print(f"{repr(item):<25} | {norm:<22} | {regra:<5} | {categoria}")

print("\n--- Simulação da Máquina de Estados (Opções: 1 -> 'papel' -> 2 -> 3) ---")
estado = "MENU"
historico = []
acoes = [("1", "papel"), ("2", None), ("3", None)]

for opcao, entrada in acoes:
    prox = proximo_estado(estado, opcao)
    print(f"Estado: {estado} --(opção {opcao})--> {prox}")
    estado = prox
    if estado == "CLASSIFICAR" and entrada is not None:
        cat, orientacao, regra = inferir(entrada)
        historico.append((entrada, cat, regra))
        print(f"  Classificado '{entrada}' => {cat} ({regra})")
        estado = proximo_estado(estado, "concluida")
        print(f"  Retorno automático => {estado}")
    elif estado == "HISTORICO":
        print(f"  Histórico em sessão: {historico}")
        estado = proximo_estado(estado, "concluida")
        print(f"  Retorno automático => {estado}")


## 5. Teste Interativo no Colab

Execute a célula abaixo para digitar qualquer resíduo e testar a classificação em tempo real.


In [ ]:
def testar_residuo():
    entrada = input("Digite o nome de um resíduo doméstico (ex: pilha, garrafa pet, borra de café): ")
    categoria, orientacao, regra = inferir(entrada)
    print(f"\nResultado: {categoria} (Regra: {regra})")
    print(f"Orientação: {orientacao}")
    print("Aviso: Recomendação automatizada por regras; confirme conforme a coleta local.")

# Descomente a linha abaixo para testar interativamente no Google Colab:
# testar_residuo()
